# 🔥 Outlier Handling Notebook (GitHub Reference Version)
Author: Sachin

Is notebook me outliers detect aur handle karne ki important techniques cover ki gayi hain:
- IQR Detection
- Z-Score Detection
- Winsorization
- Capping / Clipping
- Log Transformation
- Robust Scaling

Har code ke upar simple explanation comment diya gaya hai taaki future me quickly recall ho sake.

## 1️⃣ Sample Dataset Create Karna

In [1]:
# Libraries import kar rahe hain
import numpy as np
import pandas as pd

# Reproducibility ke liye random seed set
np.random.seed(42)

# Normal distributed data create kar rahe hain
data = np.random.normal(50,10,100)

# Artificial extreme values add kar rahe hain (ye outliers hain)
data = np.append(data,[150,170,200])

# DataFrame create kar rahe hain
df = pd.DataFrame({'feature':data})

df.head()

,feature
0,54.967142
1,48.617357
2,56.476885
3,65.230299
4,47.658466


## 2️⃣ IQR Method (Outlier Detection)

In [2]:
# Q1 aur Q3 calculate kar rahe hain
Q1 = df['feature'].quantile(0.25)
Q3 = df['feature'].quantile(0.75)

# IQR calculate kar rahe hain
IQR = Q3 - Q1

# Lower aur upper boundary define kar rahe hain
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Outliers detect kar rahe hain
outliers = df[(df['feature'] < lower) | (df['feature'] > upper)]

outliers

,feature
74,23.802549
100,150.000000
101,170.000000
102,200.000000


## 3️⃣ Z‑Score Method

In [3]:
# Mean aur standard deviation calculate kar rahe hain
mean = df['feature'].mean()
std = df['feature'].std()

# Z-score calculate kar rahe hain
df['zscore'] = (df['feature'] - mean) / std

# |z| > 3 ko outlier treat kar rahe hain
df[np.abs(df['zscore']) > 3]

,feature,zscore
100,150.0,4.214932
101,170.0,5.080279
102,200.0,6.378300


## 4️⃣ Winsorization (Important Technique)
Winsorization me extreme values ko delete nahi karte.
Unhe percentile boundary se replace kar dete hain.

Example: Top 5% aur bottom 5% values ko cap kar dete hain.

In [4]:
# Winsorization apply karne ke liye scipy library use karte hain
from scipy.stats.mstats import winsorize

# limits=[0.05,0.05] ka matlab
# lowest 5% values -> 5th percentile
# highest 5% values -> 95th percentile

winsorized_data = winsorize(df['feature'], limits=[0.05,0.05])

# Result ko DataFrame column me store kar rahe hain
df['winsorized_feature'] = winsorized_data

df.head()

,feature,zscore,winsorized_feature
0,54.967142,0.103112,54.967142
1,48.617357,-0.171627,48.617357
2,56.476885,0.168434,56.476885
3,65.230299,0.547171,65.230299
4,47.658466,-0.213115,47.658466


## 5️⃣ Capping / Clipping Using IQR

In [5]:
# Extreme values ko IQR boundary ke according clip kar rahe hain
df['capped_feature'] = df['feature'].clip(lower,upper)

df.head()

,feature,zscore,winsorized_feature,capped_feature
0,54.967142,0.103112,54.967142,54.967142
1,48.617357,-0.171627,48.617357,48.617357
2,56.476885,0.168434,56.476885,56.476885
3,65.230299,0.547171,65.230299,65.230299
4,47.658466,-0.213115,47.658466,47.658466


## 6️⃣ Log Transformation

In [6]:
# Log transformation skewed data ko stable banane me help karta hai
# log1p use kar rahe hain taaki zero values pe issue na aaye

df['log_feature'] = np.log1p(df['feature'])

df.head()

,feature,zscore,winsorized_feature,capped_feature,log_feature
0,54.967142,0.103112,54.967142,54.967142,4.024765
1,48.617357,-0.171627,48.617357,48.617357,3.904341
2,56.476885,0.168434,56.476885,56.476885,4.051383
3,65.230299,0.547171,65.230299,65.230299,4.193138
4,47.658466,-0.213115,47.658466,47.658466,3.884826


## 7️⃣ Robust Scaling

In [7]:
# RobustScaler median aur IQR use karta hai
# Isliye ye outliers ke presence me better perform karta hai

from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

df['robust_scaled'] = scaler.fit_transform(df[['feature']])

df.head()

,feature,zscore,winsorized_feature,capped_feature,log_feature,robust_scaled
0,54.967142,0.103112,54.967142,54.967142,4.024765,0.512652
1,48.617357,-0.171627,48.617357,48.617357,3.904341,-0.059722
2,56.476885,0.168434,56.476885,56.476885,4.051383,0.648741
3,65.230299,0.547171,65.230299,65.230299,4.193138,1.437780
4,47.658466,-0.213115,47.658466,47.658466,3.884826,-0.146157


## 📌 Final Quick Notes
- IQR → Skewed data detection
- Z-score → Normal distribution
- Winsorization → Extreme values ko cap karna
- Clipping → IQR boundary based capping
- Log transform → Skew reduce
- RobustScaler → Median based scaling (outlier resistant)